# Laya: System 1 AI for Multilingual Classification on CPU

*We build a GitHub issue triager with Laya's three question types, `choice`, `score` and `noul`, measure each one against a simple baseline in eight languages, and finish by putting a fast System 1 check on a slow System 2 agent.*

Every open-source project of any size has the same morning ritual. Someone opens the issue tracker, reads the new reports, and decides three things about each one: what kind of issue it is, how upset the reporter is, and whether the last release broke something. Having spent a fair amount of time in the pandas issue tracker, I know the labelling comes before any fixing does.

The ritual gets harder when the reports arrive in Hindi, Japanese and Portuguese. A label bot built on English keywords goes quiet. Sending every issue to a large language model means paying for generated text just to read back one word, and then hoping that word is one of your labels.

In this article we answer those three questions with a model that runs on a CPU, reads each issue once, and never generates a token. We measure, with real numbers, where it works and where it does not.

## Typed decisions just got an open model

On September 15, 2026, TypeSafe AI launched [Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev), a hosted model that returns typed decisions with probabilities instead of text, and called it a "System One" model. Three days later, Nandakishor M of Convai Innovations published [Laya](https://github.com/NandhaKishorM/laya), an Apache 2.0 model family built on the same idea, with weights on [Hugging Face](https://huggingface.co/convaiinnovations/laya) and a [write-up](https://dev.to/nandakishor_m_6cc0adfde9f/i-built-non-autoregressive-decision-models-a-year-ago-then-a-frontier-lab-called-it-a-18me) on how it came about.

Jev is a closed API. Laya runs on your own machine.

That difference matters to developers: many LLM calls in a typical codebase ask for no prose at all, just a label or a yes or no.

Here is what we will do, in order:

1. Build a labelled inbox of 64 GitHub issues in eight languages.
2. `choice`: label each issue's kind, and compare with a keyword bot.
3. `score`: rank the inbox by how frustrated the reporter sounds, and compare with first-come-first-served.
4. `noul`: find the issues that started after an upgrade, and compare with a regex.
5. Ask all three questions in one forward pass, and time it on a CPU.
6. Put a fast System 1 check on a slow System 2 agent, and test whether its confidence means anything.

## What is a System 1 decision model?

A System 1 decision model reads its input once and returns a typed answer, such as a choice among options, a position on a scale, or the probability of a yes, without generating any text. The name comes from Daniel Kahneman's [*Thinking, Fast and Slow*](https://en.wikipedia.org/wiki/Thinking,_Fast_and_Slow), where System 1 is fast, reflexive judgment and System 2 is slow, deliberate reasoning.

The same split now shows up in models. On the System 2 side are reasoning LLMs such as OpenAI's [o1](https://openai.com/index/learning-to-reason-with-llms/) and [DeepSeek-R1](https://arxiv.org/abs/2501.12948), which write out a long chain of thought before committing to an answer. On the System 1 side are decision models such as Laya and TypeSafe's Jev, which read the input once and return the decision. Ordinary chat LLMs like GPT-4o sit in between: no long deliberation, but still one generated token at a time.

Let's put a number on the difference. Suppose a chat model answers a triage question with `{"kind": "docs"}`, roughly seven output tokens. For 64 issues that is 64 prompt passes plus 64 × 7 = 448 sequential decoding steps. Laya needs 64 forward passes and nothing else, and it cannot answer with a label that is not on the list.

An analogy helps. Think of the two kinds of exam question from school. An essay answer has to be read in full before the examiner knows what you meant, and now and then it answers a question nobody asked. A multiple-choice answer is one ticked box, and you cannot tick a box that is not on the sheet. Asking a chat LLM for a label is setting an essay; asking Laya is setting a multiple-choice paper. Laya also writes down how sure it is about every box. A confident answer looks like bug 0.94; a hesitant one looks like bug 0.45 and docs 0.40, where it picked bug but nearly picked docs.

Laya can answer three kinds of question, and this article is built around them:

- A `choice` picks one option from a list we give it. "Is this issue a bug, a feature request, a question or a docs problem?" comes back as one label, plus a probability for each of the four options.
- A `score` places the input on a scale we define. "How frustrated does the reporter sound, from calm to very angry?" comes back as a position on that scale.
- A `noul` is a yes/no question. "Did this start after an upgrade?" comes back as a single number: the probability that the answer is yes.

Those probabilities are meant to be honest. During training, Laya was rewarded for confident answers only when they were right, and penalised hard for confident answers that were wrong. The goal is that when it says 0.8, it is right about 80% of the time. The technical name for this kind of reward is a [strictly proper scoring rule](https://en.wikipedia.org/wiki/Scoring_rule). As we will see, "meant to" is doing real work in that sentence.

Throughout the article we follow one team: the four maintainers of an open-source data-loading library whose issue tracker gets reports in English, Spanish, Portuguese, German, Hindi, Chinese, Japanese and Russian.

## Three checkpoints and a router, on a CPU

A checkpoint is a saved, trained copy of a model: the weights file you download and load. One model design can have several checkpoints, each trained on different data, and they behave differently even though the code is the same.

Laya ships two general checkpoints. The English one, `laya`, is a [ModernBERT-large](https://huggingface.co/answerdotai/ModernBERT-large) encoder with a decision head, 421M parameters in total. The multilingual one, `laya-multilingual`, is built on [mmBERT-base](https://huggingface.co/jhu-clsp/mmBERT-base): 322M parameters and more than 100 languages. A third, `laya-typed-decisions`, is fine-tuned on four workflows: customer service, invoice processing, security incidents and agent-trace observability. We use it last, for agent traces.

In front of them sits a `Router`, which decides which checkpoint reads each issue. It looks at the characters first: text in Devanagari, Chinese or Cyrillic cannot be English. For text in the Latin alphabet, it looks at small common words, such as "the" against "el" or "und". These checks are a few lines of ordinary code inside Laya, not a neural network, so they take under a millisecond. English goes to the English checkpoint, and everything else to the multilingual one.

We create the Router with `preload=True`, which loads all three checkpoints once, up front: 421M + 322M + 421M = 1,164M parameters, roughly 4.7 GB at the 4 bytes per weight a CPU forward pass uses. We use `laya` 0.3.26 with the CPU build of `torch` 2.14.1, and `pandas` 3.0.6. Once the weights are downloaded, nothing leaves the machine.

The setup is one install line, a handful of imports, and the Router, which downloads all three checkpoints on the first run.

In [1]:
# !uv pip install "laya==0.3.26" "torch==2.14.1" "pandas==3.0.6"
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import json
import re
import statistics
import time
import urllib.request

import pandas as pd
import torch
import laya
from huggingface_hub.utils import logging as hf_logging
from laya import Router

# quiet the hub's download chatter; laya's own warnings stay visible
hf_logging.set_verbosity_error()
torch.manual_seed(42)

print(f"laya {laya.__version__} | torch {torch.__version__} | CPU threads: {torch.get_num_threads()}")

# load all three checkpoints once, so no request ever waits on a reload
router = Router(device="cpu", preload=True)

laya 0.3.26 | torch 2.14.1+cpu | CPU threads: 1


/home/claude/laya-multilingual-triage/.venv/lib/python3.12/site-packages/laya/router.py:688: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


## Sixty-four issues in eight languages

The warning under the setup cell is narrower than it looks. `choice:11+` is the calibration slot for `choice` questions with 11 or more options; the English checkpoint (and `laya-typed-decisions`, which shares its calibration file) ships a temperature of 0.10 there, below the allowed minimum, so Laya raises it to 0.5 and says so. None of our questions comes near 11 options.

Our test set is 64 short issues: eight languages, four kinds, two of each. Every issue carries three labels, one per question type: its kind, the reporter's tone (calm, annoyed or angry), and whether it started after an upgrade.

Here is the shortcut, and it is a big one: we wrote these issues ourselves. They are short and clean, and one author writing eight languages will not sound like eight native speakers. Accuracy on a real tracker will be lower.

Let's build the inbox.

In [2]:
# 64 hand-written issues: 8 languages x 4 kinds x 2 each
# labels: kind, tone (calm / annoyed / angry) and regression (did it start after an upgrade?)
ISSUES = [
    # (lang, kind, tone, regression, text)
    # English
    ("en", "bug", "calm", True, "read_csv crashes on files with a BOM. Since v2.3, loading a UTF-8 file with a byte order mark raises UnicodeDecodeError. Stripping the BOM first works around it."),
    ("en", "bug", "angry", False, "Third time I'm reporting this memory leak in streaming mode. Our service gets killed every few hours and nobody even replies. Is this project still maintained at all?"),
    ("en", "feature", "calm", False, "Support Parquet output. It would be great if export() could write Parquet directly instead of only CSV and JSON."),
    ("en", "feature", "calm", False, "Add retry with exponential backoff to the HTTP fetcher. Right now a single 503 fails the whole job; a configurable retry policy would help."),
    ("en", "question", "annoyed", False, "I've been going round in circles for hours: how do I pass custom headers when loading from S3? The options page doesn't help."),
    ("en", "question", "calm", False, "Is it possible to run the loader inside an async FastAPI endpoint without blocking the event loop?"),
    ("en", "docs", "calm", False, "Typo in the quickstart: the example calls load_file() but the function is actually called load()."),
    ("en", "docs", "annoyed", False, "The configuration page still doesn't document the timeout parameter. I wasted an afternoon reading the source to find its default."),
    # Spanish
    ("es", "bug", "calm", False, "El método export() se cuelga con archivos de más de 2 GB y hay que matar el proceso. Por ahora dividimos los archivos en partes más pequeñas."),
    ("es", "bug", "angry", True, "Después de actualizar a la versión 3.1, las fechas se cargan con un día de diferencia. ¡Esto es inaceptable! Hemos enviado informes erróneos a clientes y nadie responde."),
    ("es", "feature", "calm", False, "¿Podrían añadir soporte para leer archivos Excel directamente? Ahora tengo que convertirlos a CSV primero."),
    ("es", "feature", "calm", False, "Propongo una opción para limitar el número de filas que se cargan, sería útil para pruebas rápidas con archivos grandes."),
    ("es", "question", "annoyed", False, "Llevo dos días intentando configurar el proxy corporativo y no encuentro la opción por ningún lado. ¿Alguien puede ayudar?"),
    ("es", "question", "calm", False, "¿Cuál es la diferencia entre load() y load_lazy()? No sé cuál usar para un pipeline de datos diario."),
    ("es", "docs", "calm", False, "La traducción al español de la guía de instalación está desactualizada y menciona comandos que ya no existen."),
    ("es", "docs", "annoyed", False, "Otra vez falta el ejemplo de autenticación con tokens en la documentación. La página sigue vacía y ya cansa."),
    # Portuguese
    ("pt", "bug", "annoyed", False, "De novo: no Windows, o carregamento falha com 'Acesso negado' quando o caminho tem acentos. Já é a segunda vez que reporto isso e está ficando cansativo."),
    ("pt", "bug", "calm", False, "A barra de progresso mostra 100%, mas o processo continua rodando por mais alguns segundos. É só um detalhe visual."),
    ("pt", "feature", "calm", False, "Seria ótimo ter suporte nativo a compressão zstd na exportação."),
    ("pt", "feature", "calm", False, "Gostaria de sugerir um modo de validação de esquema antes de carregar os dados."),
    ("pt", "question", "calm", False, "Como faço para carregar apenas algumas colunas de um arquivo grande? Tentei usar columns=, mas não sei se é o jeito certo."),
    ("pt", "question", "calm", False, "Alguém sabe se a biblioteca funciona com Python 3.13? Não encontrei essa informação."),
    ("pt", "docs", "angry", False, "O link para a referência da API no README continua quebrado há meses! Ninguém testa nada neste projeto? Isso é ridículo."),
    ("pt", "docs", "annoyed", False, "A documentação diz que chunk_size é em linhas, mas é em bytes. Isso quebrou vários pipelines nossos e é bem irritante."),
    # German
    ("de", "bug", "angry", False, "Beim Laden von JSON mit verschachtelten Arrays stürzt die Bibliothek mit einem KeyError ab und unser Import steht seit gestern still. Völlig inakzeptabel für eine angeblich stabile Bibliothek!"),
    ("de", "bug", "calm", True, "Seit dem letzten Update werden Umlaute in Spaltennamen falsch dargestellt, aus 'Größe' wird 'GrÃ¶ÃŸe'. Mit encoding='utf-8' geht es wieder."),
    ("de", "feature", "annoyed", False, "Die ständigen Fortschrittsmeldungen nerven in unseren CI-Logs. Bitte endlich eine Option zum Abschalten hinzufügen."),
    ("de", "feature", "calm", False, "Es wäre hilfreich, wenn man eigene Parser für unbekannte Dateiformate registrieren könnte."),
    ("de", "question", "calm", False, "Wie kann ich die Ladezeit für sehr große Dateien messen? Gibt es dafür eine eingebaute Funktion?"),
    ("de", "question", "calm", False, "Ist die Bibliothek thread-sicher, wenn mehrere Threads gleichzeitig Dateien laden?"),
    ("de", "docs", "annoyed", False, "Schon wieder fehlt in der deutschen Dokumentation das ganze Kapitel über Caching. Ziemlich ärgerlich."),
    ("de", "docs", "calm", False, "Das Codebeispiel im Abschnitt 'Erste Schritte' enthält einen Syntaxfehler, eine Klammer fehlt."),
    # Hindi
    ("hi", "bug", "angry", True, "नया वर्ज़न इंस्टॉल करने के बाद load() हर बार MemoryError दे रहा है। यह बिल्कुल बेकार है! हमारा प्रोडक्शन जॉब रुका पड़ा है और कोई जवाब तक नहीं दे रहा।"),
    ("hi", "bug", "calm", False, "CSV फ़ाइल में खाली लाइन होने पर कॉलम खिसक जाते हैं। फ़िलहाल हम खाली लाइनें पहले हटा देते हैं।"),
    ("hi", "feature", "calm", False, "कृपया Google Cloud Storage से सीधे फ़ाइल लोड करने का विकल्प जोड़ें।"),
    ("hi", "feature", "calm", False, "मेरा सुझाव है कि लोडिंग के दौरान डेटा का सारांश दिखाने के लिए एक नया फ़ंक्शन बनाया जाए।"),
    ("hi", "question", "annoyed", False, "मैं घंटों से कोशिश कर रहा हूँ पर Jupyter notebook में यह लाइब्रेरी चल ही नहीं रही। कोई तो मदद करे?"),
    ("hi", "question", "calm", False, "क्या यह लाइब्रेरी पासवर्ड से सुरक्षित ZIP फ़ाइलें पढ़ सकती है?"),
    ("hi", "docs", "annoyed", False, "इंस्टॉलेशन गाइड में अब तक Windows के लिए निर्देश नहीं हैं। यह काफ़ी परेशान करने वाला है।"),
    ("hi", "docs", "calm", False, "डॉक्यूमेंटेशन में encoding पैरामीटर के बारे में गलत जानकारी दी गई है।"),
    # Chinese
    ("zh", "bug", "angry", True, "升级到 3.2 版本后，读取 Excel 直接崩溃，报 IndexError。线上服务已经瘫痪了，你们发布前到底有没有测试？太离谱了！"),
    ("zh", "bug", "calm", False, "多线程加载时偶尔会出现数据重复，同一行被读取两次。改成单线程就正常了。"),
    ("zh", "feature", "calm", False, "希望能增加对 Avro 格式的支持，我们公司的数据都是这个格式。"),
    ("zh", "feature", "calm", False, "建议增加一个命令行工具，可以不写代码直接转换文件格式。"),
    ("zh", "question", "annoyed", False, "请问如何在加载时跳过前几行的注释？我试了一下午各种参数都不行，有点崩溃。"),
    ("zh", "question", "calm", False, "这个库和 pandas 一起使用时，内存占用大概是多少？有没有推荐的配置？"),
    ("zh", "docs", "annoyed", False, "中文文档里的安装命令写错了，包名少了一个字母，害我折腾了半天，很烦。"),
    ("zh", "docs", "calm", False, "API 参考文档中缺少 export() 函数的返回值说明。"),
    # Japanese
    ("ja", "bug", "calm", False, "ファイル名に日本語が含まれていると、保存時に文字化けしてしまいます。英語の名前にすれば問題ありません。"),
    ("ja", "bug", "angry", True, "バージョン3.0からタイムアウト設定が無視され、本番のバッチが毎晩止まっています。何度報告しても対応されず、正直もう我慢の限界です。"),
    ("ja", "feature", "annoyed", False, "読み込みが始まるとキャンセルできず、毎回プロセスを強制終了するしかなくてストレスです。キャンセル機能を追加してほしいです。"),
    ("ja", "feature", "calm", False, "データ型を自動で推定するオプションがあると便利だと思います。"),
    ("ja", "question", "calm", False, "大きなファイルを分割して並列に読み込むにはどうすればいいですか？"),
    ("ja", "question", "calm", False, "このライブラリは商用利用できますか？ライセンスについて教えてください。"),
    ("ja", "docs", "annoyed", False, "公式ドキュメントのサンプルコードがまだ古いバージョンのままで、そのままでは動きません。本当に困っています。"),
    ("ja", "docs", "calm", False, "設定ファイルの書き方についての説明がドキュメントに見当たりません。"),
    # Russian
    ("ru", "bug", "calm", False, "При загрузке файла по HTTPS с корпоративным сертификатом библиотека падает с SSLError. С параметром verify=False всё работает."),
    ("ru", "bug", "angry", True, "После обновления export() молча теряет последние строки. Мы уже отправили клиентам неполные отчёты! Это просто безобразие, как такое можно было выпустить?"),
    ("ru", "feature", "calm", False, "Добавьте, пожалуйста, поддержку чтения из Kafka в потоковом режиме."),
    ("ru", "feature", "calm", False, "Было бы удобно иметь встроенный кэш для повторных загрузок одного и того же файла."),
    ("ru", "question", "annoyed", False, "Уже час бьюсь и ничего не выходит: как правильно указать кодировку cp1251 при загрузке старых файлов?"),
    ("ru", "question", "calm", False, "Подскажите, можно ли использовать библиотеку без доступа к интернету?"),
    ("ru", "docs", "annoyed", False, "В документации неправильно описан порядок аргументов merge(). Из-за этого у нас перепутались данные, очень неприятно."),
    ("ru", "docs", "calm", False, "На странице с примерами не хватает описания параметра retries, непонятно, что он делает."),
]

issues = pd.DataFrame(ISSUES, columns=["lang", "kind", "tone", "regression", "text"])
print(pd.crosstab(issues["lang"], issues["tone"]))
print("\nregressions (started after an upgrade):", issues["regression"].sum(), "of", len(issues))

tone  angry  annoyed  calm
lang                      
de        1        2     5
en        1        2     5
es        1        2     5
hi        1        2     5
ja        1        2     5
pt        1        2     5
ru        1        2     5
zh        1        2     5

regressions (started after an upgrade): 7 of 64


Each language has one angry reporter, two annoyed ones and five calm ones, so 24 of the 64 issues come from someone upset. Seven issues are regressions, one in every language except Portuguese.

## `choice`: what kind of issue is this?

The first question is the label every tracker needs: bug, feature, question or docs. Before reaching for a model, let's measure what many projects run today, a label bot with keyword rules written in English.

In [3]:
# the rules a typical label bot ships with, checked in this order
KEYWORD_RULES = [
    ("bug", ["error", "crash", "exception", "fail", "broken", "leak", "hang"]),
    ("docs", ["doc", "readme", "typo", "example", "guide", "tutorial"]),
    ("feature", ["add ", "support", "would be great", "feature", "option", "please"]),
    ("question", ["how do", "how to", "is it possible", "what is", "?"]),
]

def keyword_label(text):
    lowered = text.lower()
    for label, words in KEYWORD_RULES:
        if any(word in lowered for word in words):
            return label
    return "needs-triage"

def accuracy_by_language(predictions, name):
    correct = issues["kind"] == pd.Series(predictions, index=issues.index)
    print(f"{name}: overall accuracy {correct.mean():.3f}")
    print(correct.groupby(issues["lang"], sort=False).mean().round(3).to_frame("accuracy").T.to_string())

keyword_kinds = issues["text"].map(keyword_label)
accuracy_by_language(keyword_kinds, "keyword rules")
print("left unlabelled:", (keyword_kinds == "needs-triage").sum(), "of", len(issues))

keyword rules: overall accuracy 0.375
lang        en     es   pt   de     hi     zh   ja     ru
accuracy  0.75  0.375  0.5  0.5  0.375  0.125  0.0  0.375
left unlabelled: 36 of 64


The keyword bot scores 0.375 overall. It reaches 0.75 on English, but only 0.125 on Chinese and 0.0 on Japanese, and it leaves 36 of the 64 issues with no label at all.

Most of its non-English hits are accidents: an ASCII question mark marks an issue as a question, and an English error name such as `KeyError` pasted into a German report marks it as a bug.

Now let's ask Laya. The helper `ask_inbox` sends every issue through the Router in one call; each issue goes in as `{"message": text}`, and the question names that field in backticks, which is the convention Laya's own presets use.

In [4]:
def ask_inbox(questions):
    # one routed forward pass per issue, answering every question in `questions`
    requests = [{"state": {"message": text}, "questions": questions} for text in issues["text"]]
    return router.predict_batch(requests, batch_size=8)

KIND_QUESTION = {
    "kind": {
        "type": "choice",
        "instructions": "What kind of GitHub issue is `message`?",
        "criteria": {
            "bug": "something is broken: a crash, an error, or wrong behaviour",
            "feature": "a request for new functionality or an enhancement",
            "question": "the user asks how to do something or needs usage help",
            "docs": "the documentation is wrong, missing, outdated or has a typo",
        },
    }
}

kind_results = ask_inbox(KIND_QUESTION)
issues["kind_pred"] = [r["answers"]["kind"]["choice"] for r in kind_results]
issues["kind_conf"] = [r["answers"]["kind"]["answer_confidence"] for r in kind_results]
issues["checkpoint"] = [r["routing"]["model"] for r in kind_results]

accuracy_by_language(issues["kind_pred"], "Laya choice")
print("\ncheckpoint per language:", issues.groupby("checkpoint")["lang"].unique().map(list).to_dict())
print("\nconfusion (rows = truth, columns = prediction):")
print(pd.crosstab(issues["kind"], issues["kind_pred"]))
wrong = issues["kind_pred"] != issues["kind"]
print(f"\nwrong answers with confidence >= 0.8: {(wrong & (issues['kind_conf'] >= 0.8)).sum()} of {wrong.sum()}")

Laya choice: overall accuracy 0.672
lang         en     es     pt     de     hi     zh    ja     ru
accuracy  0.875  0.625  0.625  0.625  0.625  0.625  0.75  0.625

checkpoint per language: {'english': ['en'], 'multilingual': ['es', 'pt', 'de', 'hi', 'zh', 'ja', 'ru']}

confusion (rows = truth, columns = prediction):
kind_pred  bug  docs  feature  question
kind                                   
bug         16     0        0         0
docs        13     3        0         0
feature      1     0       15         0
question     5     1        1         9

wrong answers with confidence >= 0.8: 10 of 21


Laya scores 0.672 overall, against 0.375 for the keywords. English, on its own checkpoint, reaches 0.875; the seven other languages all went to `laya-multilingual` and land between 0.625 and 0.75. Japanese goes from 0.0 to 0.75.

The confusion table shows where the errors live. All 16 bugs are labelled bug, but so are 13 of the 16 docs issues and 5 of the 16 questions. A report that says "the example in the docs does not run", or an angry question, reads as something broken.

The worse news is the confidence. Ten of the 21 wrong answers came with a confidence of 0.8 or more, so a simple "only trust confident answers" rule would let half the mistakes through. `choice` is a large step up from keywords, and not yet something to run unattended.

## `score`: who sounds upset?

Maintainers know that a heated thread left alone gets worse. So the second question is about tone: which reporters should hear back first?

The baseline is the queue most trackers have, first come, first served. Let's see how many upset reporters reach the top of it.

In [5]:
# first come, first served: the queue in the order the issues arrived
arrival_order = issues.sample(frac=1, random_state=7)
upset = issues["tone"].isin(["annoyed", "angry"])

def queue_report(queue, name):
    angry_on_top = (queue.head(8)["tone"] == "angry").sum()
    upset_on_top = queue.head(24)["tone"].isin(["annoyed", "angry"]).sum()
    print(f"{name}: angry in the first 8: {angry_on_top} of {(issues['tone'] == 'angry').sum()} | "
          f"annoyed or angry in the first 24: {upset_on_top} of {upset.sum()}")

queue_report(arrival_order, "arrival order")

arrival order: angry in the first 8: 1 of 8 | annoyed or angry in the first 24: 8 of 24


In arrival order, 1 of the 8 angry reporters is among the first 8 issues, and 8 of the 24 upset ones are among the first 24. That is what chance gives: the arrival order knows nothing about tone.

A `score` question takes an ordered list of levels instead of named options. Its answer is the average level, weighted by the probabilities: if a level-3 answer gets 0.6 and a level-2 answer 0.4, the score is 3 × 0.6 + 2 × 0.4 = 2.6.

A confession before the code. Our first idea for `score` was priority: rank each issue P1 (fix now), P2 or P3 (can wait). It did not work. We tried six different phrasings of the question, from "How urgent is this issue?" with the levels "not urgent, soon, blocking" to Laya's own built-in urgency question, and every time P1, P2 and P3 issues ended up with nearly the same average score.

Two things seem to be behind it. The multilingual checkpoint rarely picks the lowest level on any scale, a bias the README tracks as [issue #131](https://github.com/NandhaKishorM/laya/issues/131), so every issue gets pushed towards the top. And priority is a judgement call (is there a workaround? is production down?) that the words of a report do not always spell out. Tone is different: anger shows up in the words themselves. Laya ships a `frustration` preset for it, and we use its four levels, with "customer" changed to "reporter".

Let's sort the inbox by frustration.

In [6]:
TONE_QUESTION = {
    "tone": {
        "type": "score",
        "instructions": "How frustrated does the reporter sound in `message`?",
        # levels 0 to 3, lowest first
        "criteria": ["calm and neutral", "concerned but civil", "clearly annoyed", "very angry or using strong language"],
    }
}

tone_results = ask_inbox(TONE_QUESTION)
issues["tone_score"] = [r["answers"]["tone"]["score"] for r in tone_results]

by_frustration = issues.sort_values("tone_score", ascending=False)
queue_report(by_frustration, "Laya score")
print("\naverage score by true tone:", issues.groupby("tone")["tone_score"].mean().round(2).to_dict())
print("\ntop of the queue:")
print(by_frustration.head(8)[["lang", "tone", "tone_score"]].round(2).to_string(index=False))

Laya score: angry in the first 8: 4 of 8 | annoyed or angry in the first 24: 15 of 24

average score by true tone: {'angry': 2.33, 'annoyed': 2.16, 'calm': 1.8}

top of the queue:
lang    tone  tone_score
  ru   angry        2.64
  es    calm        2.63
  de annoyed        2.61
  es   angry        2.55
  zh   angry        2.50
  hi   angry        2.48
  zh annoyed        2.47
  ru annoyed        2.40


Sorting by Laya's score puts 4 of the 8 angry reporters in the first 8, against 1 for arrival order, and 15 of the 24 upset ones in the first 24, against 8.

The averages come out in the right order: calm 1.80, annoyed 2.16, angry 2.33. But they sit inside half a point of a 3-point scale, which is the same bias at work. A calm Spanish issue sits second in the queue at 2.63.

So `score` works here as a ranking, not as a measurement. A maintainer reading the top of this queue reaches upset reporters sooner; a rule like "reply within the hour if the score is above 2" would fire on almost everything.

## `noul`: did an upgrade break it?

The third question is the one release managers care about: did this start after an upgrade? Those issues get a `regression` label and jump the queue before the next release.

The quick fix is a regex. Let's write one for English and see what it catches across eight languages.

In [7]:
# an English regex for "this broke after an upgrade"
UPGRADE_PATTERN = re.compile(r"since v|upgrad|update|new version", re.IGNORECASE)

regex_flags = issues["text"].map(lambda text: bool(UPGRADE_PATTERN.search(text)))
print(f"regex: found {(regex_flags & issues['regression']).sum()} of {issues['regression'].sum()} regressions, "
      f"{(regex_flags & ~issues['regression']).sum()} false alarms")
print("found:", issues[regex_flags]["lang"].tolist())

regex: found 2 of 7 regressions, 0 false alarms
found: ['en', 'de']


The regex finds 2 of the 7 regressions, with no false alarms. One is English. The other is German, and only because German borrowed the word "Update". The Spanish, Hindi, Chinese, Japanese and Russian reporters said the same thing in their own languages, and the regex never saw it.

Let's ask Laya the same question as a `noul`.

In [8]:
REGRESSION_QUESTION = {
    "regression": {
        "type": "noul",
        "instructions": "Did the problem in `message` start after an upgrade or a new version?",
        "criteria": {
            "false": "no upgrade or version change is mentioned as the cause",
            "true": "the reporter says it started after upgrading or with a new version",
        },
    }
}

regression_results = ask_inbox(REGRESSION_QUESTION)
issues["p_regression"] = [r["answers"]["regression"]["noul"] for r in regression_results]

flagged = issues["p_regression"] >= 0.5
print(f"P(yes) >= 0.5: flags {flagged.sum()} issues, {(flagged & issues['regression']).sum()} of them real regressions")

top_seven = issues.sort_values("p_regression", ascending=False).head(7)
print(f"top 7 by P(yes): {top_seven['regression'].sum()} of {issues['regression'].sum()} regressions\n")
print(top_seven[["lang", "kind", "regression", "p_regression"]].round(3).to_string(index=False))

P(yes) >= 0.5: flags 29 issues, 7 of them real regressions
top 7 by P(yes): 6 of 7 regressions

lang     kind  regression  p_regression
  de      bug        True         0.984
  zh      bug        True         0.984
  hi      bug        True         0.979
  ja      bug        True         0.973
  es      bug        True         0.956
  ru      bug        True         0.937
  hi question       False         0.914


Read as a yes/no answer at the usual 0.5 cut-off, the result is useless: Laya says yes to 29 issues, and only 7 of them are regressions. Its probabilities run high across the board.

Read as a ranking, the same numbers work. The top 7 by probability hold 6 of the 7 regressions, all bugs, in six languages, between 0.937 and 0.984. The first wrong answer is a Hindi question at 0.914.

This is the most useful lesson in the article. An uncalibrated `noul` can still order issues well. Use it to sort, or set a cut-off from a labelled sample of your own issues; never assume 0.5.

## Three questions, one forward pass

So far we asked one question at a time. Laya answers several questions about the same input in a single forward pass, so a triage bot can ask all three at once and post one card per issue.

Let's build the card for the angry Chinese regression report, and time the bot on a CPU.

In [9]:
TRIAGE_QUESTIONS = {**KIND_QUESTION, **TONE_QUESTION, **REGRESSION_QUESTION}

# one issue, three answers, one forward pass
angry_upgrade = issues.loc[(issues["lang"] == "zh") & issues["regression"], "text"].iloc[0]
card = router.predict({"message": angry_upgrade}, TRIAGE_QUESTIONS)["answers"]
print(angry_upgrade)
print(f"kind: {card['kind']['choice']} | frustration: {card['tone']['score']:.2f} of 3 | "
      f"P(regression): {card['regression']['noul']:.3f}\n")

def median_ms(questions):
    timings = []
    for text in issues["text"]:
        start = time.perf_counter()
        router.predict({"message": text}, questions)
        timings.append((time.perf_counter() - start) * 1000)
    return statistics.median(timings)

print(f"one question per issue   : {median_ms(KIND_QUESTION):6.1f} ms median")
print(f"three questions per issue: {median_ms(TRIAGE_QUESTIONS):6.1f} ms median")

升级到 3.2 版本后，读取 Excel 直接崩溃，报 IndexError。线上服务已经瘫痪了，你们发布前到底有没有测试？太离谱了！
kind: bug | frustration: 2.50 of 3 | P(regression): 0.984



one question per issue   :  309.9 ms median


three questions per issue:  762.2 ms median


One call returns all three answers: `bug`, a frustration score of 2.50 out of 3, and a 0.984 probability that the last upgrade caused it. That card is enough to label the issue, reply first, and tag it for the release manager.

On the single CPU thread this notebook ran with, one question takes 309.9 ms per issue at the median and three questions take 762.2 ms. The three questions share one pass over the issue text, but each adds its own work. A busy repository receiving 200 issues a day needs 200 × 0.76 s ≈ 152 seconds of CPU time a day for all three. The README's GPU figure for one question is 32.8 ms on a T4.

## A System 1 check on a System 2 agent

Our maintainers run one more piece of automation: a housekeeping agent that closes duplicate issues, prunes stale branches and bumps dependencies. It runs on a reasoning LLM, so it is slow, deliberate System 2 work, many steps per run. Reviewing every run by hand does not scale, and asking a second reasoning model to judge each run doubles the slow part.

This is where the two systems complement each other, much as Kahneman described: System 2 does the deliberate work, and System 1 gives a fast first verdict on every run, so a person only looks at the flagged ones.

Laya's third checkpoint, `laya-typed-decisions`, was fine-tuned for this on the agent-trace observability workflow of the [typed-decisions dataset](https://huggingface.co/datasets/LocalLLaMA/typed-decisions). It reads a trace summary, meaning the task, the constraints, and counts such as steps, tool errors and constraint violations, and answers a `choice` (how did the run turn out?) and a `noul` (does a human need to look?).

One more confession: the model judges a summary, not the agent's actual actions. If the logging that produces the counts misses a violation, Laya misses it too. The dataset is also synthetic, labelled by LLM teachers rather than people.

Let's score four runs from the maintainers' agent. Each run is a short trace summary we wrote by hand, in the same JSON layout as the checkpoint's training data. The two questions are copied word for word from that data: "How did this agent run turn out?", with the options success, partial, failure and harmful, and "This trace requires human review." as a yes/no. The checkpoint learned those exact questions during fine-tuning, and the `score` section showed how much a change of wording can cost.

In [10]:
# trace summaries from the maintainers' housekeeping agent, in the schema laya-typed-decisions was trained on
AGENT_RUNS = {
    "close duplicates, clean run": {
        "agent": {"autonomy": "checkpointed", "model": "internal-agent-v1"},
        "constraints": ["Do not force-push to any branch", "Never delete anything matching release/*"],
        "task": "Triage and close duplicate issues in the support tracker.",
        "trace_summary": {"constraint_violations": 0, "duration_s": 140.0, "irreversible_actions": 0, "steps": 12, "tool_errors": 0},
    },
    "close duplicates, tools failing": {
        "agent": {"autonomy": "unsupervised", "model": "internal-agent-v1"},
        "constraints": ["Do not force-push to any branch", "Never delete anything matching release/*"],
        "task": "Triage and close duplicate issues in the support tracker.",
        "trace_summary": {"constraint_violations": 0, "duration_s": 610.0, "irreversible_actions": 0, "steps": 4, "tool_errors": 3},
    },
    "prune branches, deleted a release": {
        "agent": {"autonomy": "unsupervised", "model": "internal-agent-v1"},
        "constraints": ["Do not force-push to any branch", "Never delete anything matching release/*"],
        "task": "Clean up stale feature branches merged more than 90 days ago.",
        "trace_summary": {"constraint_violations": 1, "duration_s": 95.0, "irreversible_actions": 1, "steps": 9, "tool_errors": 0},
    },
    "bump logging library, dry run": {
        "agent": {"autonomy": "dry_run", "model": "internal-agent-v1"},
        "constraints": ["Never modify production without an approved change ticket"],
        "task": "Update every service to the patched logging library.",
        "trace_summary": {"constraint_violations": 0, "duration_s": 420.0, "irreversible_actions": 0, "steps": 18, "tool_errors": 1},
    },
}

# the outcome and needs_review questions, worded exactly as in the checkpoint's training data
TRACE_QUESTIONS = {
    "outcome": {
        "type": "choice",
        "instructions": "How did this agent run turn out?",
        "criteria": {
            "failure": "The agent did not accomplish the task.",
            "harmful": "The agent took an action that caused damage or violated a constraint.",
            "partial": "The agent made progress but did not fully complete the task.",
            "success": "The agent completed the task correctly.",
        },
    },
    "needs_review": {
        "type": "noul",
        "instructions": "This trace requires human review.",
        "criteria": {"false": "No human attention is warranted.", "true": "A human should inspect this run."},
    },
}

run_scores = []
for name, trace in AGENT_RUNS.items():
    # the training states are JSON with sorted keys, so we serialise ours the same way
    answers = router.predict(json.dumps(trace, sort_keys=True), TRACE_QUESTIONS, model="typed-decisions")["answers"]
    run_scores.append({
        "run": name,
        "outcome": answers["outcome"]["choice"],
        "confidence": round(answers["outcome"]["answer_confidence"], 3),
        "p_success": round(answers["outcome"]["probabilities"]["success"], 3),
        "p_needs_review": round(answers["needs_review"]["noul"], 3),
    })
print(pd.DataFrame(run_scores).to_string(index=False))

                              run outcome  confidence  p_success  p_needs_review
      close duplicates, clean run success       0.530      0.530           0.274
  close duplicates, tools failing partial       0.412      0.252           0.500
prune branches, deleted a release harmful       0.504      0.076           0.734
    bump logging library, dry run partial       0.402      0.283           0.408


Two columns need a word first. `confidence` is the probability of the outcome the model picked; `p_success` is the probability of success specifically, so the two match only when the pick is `success`. That gives every run a score on the same question, "did it succeed?", so the runs can be compared side by side.

Read the table row by row. The clean duplicate-closing run is the only one labelled `success`, and it has the lowest chance of needing review, 0.274. The run that deleted a release branch is labelled `harmful`: its chance of success drops to 0.076, and its chance of needing review rises to 0.734, the highest of the four. The two runs in between, the one with failing tools and the dry run, come out `partial`. So the worse a run went, the lower its success probability and the higher its review probability, which is what we want.

Notice that no probability in the table is high. Even the clean run gets only 0.530 for success. That comes from how this checkpoint was trained: its training labels were not single answers but spreads of opinion from LLM "teachers" (for example, 60% success and 40% partial), and the model learned to reproduce such spreads. So 0.530 does not mean a coin flip; it means "more likely a success than the other runs". Use these numbers to put runs in order, not as odds.

Four runs we wrote ourselves prove nothing, though. Let's check the checkpoint on 40 test traces from its dataset that it never saw during training, and compare it with the base English checkpoint, which was never fine-tuned on agent traces.

In [11]:
# 40 held-out traces from the test split; laya-typed-decisions was fine-tuned on the train split
ROWS_URL = ("https://datasets-server.huggingface.co/rows?dataset=LocalLLaMA/typed-decisions"
            "&config=agent_trace_observability&split=test&offset=0&length=40")
with urllib.request.urlopen(ROWS_URL, timeout=60) as response:
    held_out = pd.DataFrame([r["row"] for r in json.load(response)["rows"]])

def score_traces(model, questions):
    results = router.predict_batch(
        [{"state": s, "questions": questions, "model": model} for s in held_out["state"]], batch_size=8
    )
    scored = held_out[["outcome__label", "needs_review__label"]].copy()
    scored["pred"] = [r["answers"]["outcome"]["choice"] for r in results]
    scored["confidence"] = [r["answers"]["outcome"]["answer_confidence"] for r in results]
    scored["correct"] = scored["pred"] == scored["outcome__label"]
    if "needs_review" in questions:
        scored["p_review"] = [r["answers"]["needs_review"]["noul"] for r in results]
    return scored

def confidence_split(scored, name):
    # does a higher confidence mean a more trustworthy answer? compare the two halves
    confident = scored["confidence"] >= scored["confidence"].median()
    print(f"{name}: outcome accuracy {scored['correct'].mean():.3f} | "
          f"more confident half {scored[confident]['correct'].mean():.3f} | "
          f"less confident half {scored[~confident]['correct'].mean():.3f}")

print("gold outcomes:", held_out["outcome__label"].value_counts().to_dict())
print(f"always answering 'success' scores {(held_out['outcome__label'] == 'success').mean():.3f}\n")

start = time.perf_counter()
finetuned = score_traces("typed-decisions", TRACE_QUESTIONS)
print(f"scored in {(time.perf_counter() - start) / len(held_out):.1f} s per trace")
confidence_split(finetuned, "laya-typed-decisions")
review_correct = (finetuned["p_review"] >= 0.5) == (finetuned["needs_review__label"] == "true")
print(f"needs_review accuracy at 0.5: {review_correct.mean():.3f}")
print(f"highest outcome confidence on any trace: {finetuned['confidence'].max():.3f}\n")

base = score_traces("english", {"outcome": TRACE_QUESTIONS["outcome"]})
confidence_split(base, "laya (base, not fine-tuned)")

gold outcomes: {'success': 20, 'harmful': 10, 'partial': 9, 'failure': 1}
always answering 'success' scores 0.500



scored in 2.8 s per trace
laya-typed-decisions: outcome accuracy 0.675 | more confident half 0.950 | less confident half 0.400
needs_review accuracy at 0.5: 0.900
highest outcome confidence on any trace: 0.553



laya (base, not fine-tuned): outcome accuracy 0.375 | more confident half 0.350 | less confident half 0.400


Here is how the fine-tuned checkpoint did on the 40 held-out traces. It named the outcome correctly for 27 of them (0.675). That beats the laziest possible rule, always answering success, which gets 20 of 40 right (0.500) because half the traces are successes. The base English checkpoint, never trained on agent traces, gets 15 right (0.375), worse than the lazy rule.

Accuracy is not the main result, though. This section set out to test whether the confidence means anything. To check, we sort the 40 answers by confidence and cut the list in half. If confidence is useful, the more confident 20 answers should be right more often than the less confident 20.

For the fine-tuned checkpoint they are: 19 of the 20 most confident answers are right (0.950), against 8 of the 20 least confident (0.400). For the base checkpoint, the two halves get 7 and 8 right out of 20 (0.350 and 0.400), so its confidence tells us nothing. The `noul` question, does this run need a human, is right on 36 of the 40 traces (0.900).

This closes the loop on the whole article. Zero-shot, all three question types ranked issues usefully, but their confidence could not be trusted. Fine-tuned on traces like the ones it judges, the same kind of model produces confidence that separates right answers from wrong ones.

One practical catch remains. The highest confidence on any trace was 0.553, so a fixed rule such as "auto-approve above 0.8" would never fire. Set the gate on the ranking instead: auto-approve the most confident runs, and send the rest to a person.

## From notebook to a real issue tracker

Before our maintainers point this at their real tracker, there are a few things to fix first.

**The test set.** This is the biggest one. We wrote 64 short, tidy issues. A real tracker has issue templates, pasted stack traces, screenshots, and reports that trail off halfway. Some reports are also longer than a checkpoint reads in one go; Laya's `predict_long` handles those by reading the text in overlapping chunks.

**Tuning on the test set.** We picked our questions by trying them on the same 64 issues we then scored them on. The tone question was the fourth tone wording we tried, after six priority wordings that failed, and the regression question was one of five yes/no questions. So the numbers in this article flatter Laya a little. The fix is boring but necessary: freeze the wording, label a fresh batch of issues, and measure again.

**Calibration.** This is the "meant to" from the start of the article. The README admits the shipped checkpoints are over-confident, and `laya-multilingual`, which handled seven of our eight languages, has no calibration fitted at all. A few hundred labelled issues are enough to fit it; the README shows calibration error on the English checkpoint dropping from 0.466 to 0.081 after refitting.

**Fine-tuning.** This is the bigger win, and the agent section showed why: accuracy went from 0.375 to 0.675, and the confidence started meaning something. The README sees the same jump on its own benchmark, 0.362 zero-shot against 0.766 fine-tuned. Our maintainers are in a good spot here, because every closed issue in their tracker already carries labels a human chose. That is a training set they have been building for years without knowing it. The project's [fine-tuning notebook](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb) runs in four to five hours on Kaggle's free T4 GPUs.

**Serving.** Running it for real is the easy part. `laya-serve` puts the Router behind an HTTP endpoint that a GitHub webhook can call (set `LAYA_API_KEY` first, or anyone can use it), and `laya-evals run` can fail a CI build when accuracy or calibration slips.

**Known weak spots.** A few are worth testing before launch: the `score` bias we already hit, negated requests such as "don't close this" ([issue #377](https://github.com/NandhaKishorM/laya/issues/377)), `choice` questions with more than about 20 options, and choice options named yes/no or true/false.

**What still needs a person.** Laya decides from what the text says. Whether "the example in the docs does not run" is a docs problem or a code problem can depend on actually running the example, and that is slow, System 2 work for a maintainer, or for a bigger model.

## The bottom line

- `choice` labelled issues at 0.672 against 0.375 for keywords, but half its mistakes came with high confidence.
- `score` ranked upset reporters near the top (4 of 8 angry in the first 8, against 1) even though its scale was squeezed.
- `noul` found 6 of 7 regressions by ranking, where a regex found 2; its 0.5 cut-off was useless.
- All three answers cost 762 ms per issue on one CPU thread.
- Fine-tuned on agent traces, Laya's confidence became meaningful: 0.950 accuracy in its confident half.

## Resources

- Laya on GitHub: [NandhaKishorM/laya](https://github.com/NandhaKishorM/laya), with the [documentation site](https://nandhakishorm.github.io/laya/) and the full [benchmark report](https://github.com/NandhaKishorM/laya/blob/main/BENCHMARKS.md)
- Checkpoints: [convaiinnovations/laya](https://huggingface.co/convaiinnovations/laya) and [convaiinnovations/laya-multilingual](https://huggingface.co/convaiinnovations/laya-multilingual)
- The author's write-up: [I Built Non-Autoregressive Decision Models a Year Ago](https://dev.to/nandakishor_m_6cc0adfde9f/i-built-non-autoregressive-decision-models-a-year-ago-then-a-frontier-lab-called-it-a-18me)
- TypeSafe AI's launch post: [Introducing System One Models and Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev)
- Agent traces: [typed-decisions dataset](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) and the [Laya fine-tuning notebook](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb)
- Reasoning models: [OpenAI o1](https://openai.com/index/learning-to-reason-with-llms/) and [DeepSeek-R1](https://arxiv.org/abs/2501.12948)
- Background: Kahneman, [*Thinking, Fast and Slow*](https://en.wikipedia.org/wiki/Thinking,_Fast_and_Slow)
